In [38]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from pathlib import Path



In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [40]:
from pathlib import Path
test_root = Path("../data/test")

test_classes = sorted([
    folder.name
    for folder in test_root.iterdir()
    if folder.is_dir()
])

print("test classes:")
for i ,class_name in enumerate(test_classes):
    print(i,"-->",class_name)

test classes:
0 --> ambulance
1 --> autobus
2 --> kamyun
3 --> kamyunet
4 --> minibus
5 --> savari
6 --> taxi
7 --> vanet


In [41]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



In [42]:
model = models.resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    8
)

model = model.to(device)



In [43]:
checkpoint_path = "../models/resnet18_layer3_layer4_norm_aug_best.pth"

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model.eval()



ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [44]:
test_root = Path("../data/test")

test_image_paths = []
test_labels = []

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(test_classes)
}

for class_name in test_classes:
    class_dir = test_root / class_name
    label = class_to_idx[class_name]

    for image_path in class_dir.iterdir():
        if image_path.is_file():
            test_image_paths.append(str(image_path))
            test_labels.append(label)

print("Number of test images:", len(test_image_paths))
print("Number of labels:", len(test_labels))

Number of test images: 400
Number of labels: 400


In [45]:
from torch.utils.data import Dataset, DataLoader


class VehicleDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")
        label = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)

        return image, label


test_dataset = VehicleDataset(
    test_image_paths,
    test_labels,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Test DataLoader created successfully")
print("Number of batches:", len(test_loader))

Test DataLoader created successfully
Number of batches: 13


In [46]:
criterion = nn.CrossEntropyLoss()

model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_loss /= total
test_accuracy = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.2291
Test Accuracy: 94.25%


In [47]:
from sklearn.metrics import classification_report

all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=test_classes,
        digits=4
    )
)

              precision    recall  f1-score   support

   ambulance     0.9792    0.9400    0.9592        50
     autobus     0.9608    0.9800    0.9703        50
      kamyun     1.0000    0.7400    0.8506        50
    kamyunet     0.8305    0.9800    0.8991        50
     minibus     0.9423    0.9800    0.9608        50
      savari     0.9091    1.0000    0.9524        50
        taxi     1.0000    0.9600    0.9796        50
       vanet     0.9600    0.9600    0.9600        50

    accuracy                         0.9425       400
   macro avg     0.9477    0.9425    0.9415       400
weighted avg     0.9477    0.9425    0.9415       400



In [48]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_predictions)

print(cm)

[[47  0  0  1  0  0  0  2]
 [ 0 49  0  0  1  0  0  0]
 [ 1  2 37  8  2  0  0  0]
 [ 0  0  0 49  0  1  0  0]
 [ 0  0  0  1 49  0  0  0]
 [ 0  0  0  0  0 50  0  0]
 [ 0  0  0  0  0  2 48  0]
 [ 0  0  0  0  0  2  0 48]]


In [49]:
def predict_single_image(image_path, model):

    image = Image.open(image_path).convert("RGB")

    image_tensor = test_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    results = sorted(
        zip(test_classes, probabilities.cpu().tolist()),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\nImage: {image_path}")
    print("-" * 35)

    for class_name, probability in results:
        print(f"{class_name:10s}: {probability * 100:.2f}%")

    print("-" * 35)
    print(f"Prediction: {results[0][0]}")

In [50]:
predict_single_image(
    r"..\data\unclean - merge\neysan\217889120.jpg",
    model
)


Image: ..\data\unclean - merge\neysan\217889120.jpg
-----------------------------------
kamyun    : 51.34%
vanet     : 46.73%
autobus   : 0.81%
kamyunet  : 0.57%
minibus   : 0.25%
ambulance : 0.19%
savari    : 0.07%
taxi      : 0.04%
-----------------------------------
Prediction: kamyun


In [51]:
unclean_root = Path("../data/unclean - merge")
neysan_dir = unclean_root / "neysan"

neysan_images = [
    path for path in neysan_dir.iterdir()
    if path.is_file()
]

print("Number of neysan images:", len(neysan_images))

Number of neysan images: 250


In [52]:
from collections import Counter

neysan_predictions = []

for image_path in neysan_images:
    image = Image.open(image_path).convert("RGB")
    image_tensor = test_transform(image).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        outputs = model(image_tensor)
        prediction = outputs.argmax(dim=1).item()

    predicted_class = test_classes[prediction]
    neysan_predictions.append(predicted_class)

    print(f"{image_path.name} -> {predicted_class}")

print("\nPrediction summary:")
print(Counter(neysan_predictions))

194960996.jpg -> vanet
195132617.jpg -> vanet
195147641.jpg -> vanet
195148040.jpg -> vanet
195151211.jpg -> vanet
195158727.jpg -> vanet
195165208.jpg -> vanet
195371292.jpg -> vanet
195391348.jpg -> vanet
195395611.jpg -> vanet
195451128.jpg -> vanet
195478008.jpg -> vanet
195575409.jpg -> vanet
195646146.jpg -> vanet
195657144.jpg -> vanet
195718438.jpg -> vanet
195728454.jpg -> vanet
195895243.jpg -> vanet
195950402.jpg -> vanet
195955288.jpg -> vanet
195988970.jpg -> vanet
195991462.jpg -> vanet
196296699.jpg -> vanet
196501322.jpg -> vanet
196541912.jpg -> vanet
196586673.jpg -> vanet
196943968.jpg -> vanet
196946016.jpg -> vanet
196993529.jpg -> vanet
197037020.jpg -> vanet
197126230.jpg -> vanet
197155144.jpg -> vanet
197223393.jpg -> vanet
197237993.jpg -> vanet
197248545.jpg -> vanet
197253666.jpg -> vanet
197255026.jpg -> vanet
197261582.jpg -> vanet
197308030.jpg -> vanet
197319173.jpg -> vanet
197375182.jpg -> vanet
197417821.jpg -> vanet
197527148.jpg -> vanet
197629171.j

In [54]:
# Analyze confidence and threshold on neysan
import pandas as pd
selected_threshold = 0.70

neysan_results = []

model.eval()

for image_path in neysan_images:

    image = Image.open(image_path).convert("RGB")

    image_tensor = (
        test_transform(image)
        .unsqueeze(0)
        .to(device)
    )

    with torch.no_grad():

        outputs = model(image_tensor)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )[0]

        confidence, prediction = probabilities.max(
            dim=0
        )

    predicted_class = test_classes[prediction.item()]

    needs_review = confidence.item() < selected_threshold

    neysan_results.append({
        "image_path": str(image_path),
        "predicted_class": predicted_class,
        "confidence": float(confidence.item()),
        "needs_review": bool(needs_review)
    })

neysan_df = pd.DataFrame(neysan_results)

print("Total neysan images:", len(neysan_df))
print("Selected threshold:", selected_threshold)

print(
    "\nPrediction distribution:"
)

print(
    neysan_df["predicted_class"].value_counts()
)

print(
    "\nNeeds review:"
)

print(
    neysan_df["needs_review"].value_counts()
)

Total neysan images: 250
Selected threshold: 0.7

Prediction distribution:
predicted_class
vanet      248
autobus      1
kamyun       1
Name: count, dtype: int64

Needs review:
needs_review
False    242
True       8
Name: count, dtype: int64


In [55]:
# Unclean / Neysan summary

summary_neysan = pd.DataFrame({
    "Count": neysan_df["predicted_class"].value_counts(),
    "Percentage": (
        neysan_df["predicted_class"]
        .value_counts(normalize=True) * 100
    ).round(2)
})

display(summary_neysan)

review_count = neysan_df["needs_review"].sum()

print(
    f"\nNeeds review: {review_count} "
    f"({100 * review_count / len(neysan_df):.2f}%)"
)

,Count,Percentage
predicted_class,,
vanet,248,99.2
autobus,1,0.4
kamyun,1,0.4



Needs review: 8 (3.20%)


In [56]:
# Save final checkpoint with metadata

class_mapping = {
    idx: class_name
    for idx, class_name in enumerate(test_classes)
}

checkpoint = {
    "model_state_dict": model.state_dict(),

    "architecture": "resnet18",

    "class_mapping": class_mapping,

    "transforms": {
        "resize": [224, 224],
        "normalize": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225]
        },
        "inference_augmentation": "none"
    },

    "selected_threshold": 0.70,

    "seed": 42,

    "configuration": {
        "trainable_layers": [
            "layer3",
            "layer4",
            "fc"
        ],
        "optimizer": "Adam",
        "learning_rate": 2e-5,
        "epochs": 15,
        "loss": "CrossEntropyLoss",
        "pretrained": True
    }
}

final_checkpoint_path = "../models/resnet18_final_with_metadata.pth"

torch.save(
    checkpoint,
    final_checkpoint_path
)

print("Final checkpoint saved:")
print(final_checkpoint_path)

Final checkpoint saved:
../models/resnet18_final_with_metadata.pth


In [57]:
# Verify saved checkpoint

loaded_checkpoint = torch.load(
    "../models/resnet18_final_with_metadata.pth",
    map_location="cpu"
)

print("Checkpoint keys:")
print(loaded_checkpoint.keys())

print("\nSelected threshold:")
print(loaded_checkpoint["selected_threshold"])

print("\nClass mapping:")
print(loaded_checkpoint["class_mapping"])

Checkpoint keys:
dict_keys(['model_state_dict', 'architecture', 'class_mapping', 'transforms', 'selected_threshold', 'seed', 'configuration'])

Selected threshold:
0.7

Class mapping:
{0: 'ambulance', 1: 'autobus', 2: 'kamyun', 3: 'kamyunet', 4: 'minibus', 5: 'savari', 6: 'taxi', 7: 'vanet'}
